In [79]:
import pandas as pd
data = pd.read_csv("default.csv",header=None)

In [1]:
import rasterio
dataset = rasterio.open('Users/jbms/TI/Ciechanow/test.tif')

ImportError: rasterio._base does not export expected C function osr_set_traditional_axis_mapping_strategy

In [2]:
import shapely
from shapely.geometry.point import Point

In [5]:
p = Point(52.3647,4.9469)

In [14]:
circle = p.buffer(.00001)

In [23]:
xyz = list(circle.bounds)

In [28]:
xyz

TypeError: list indices must be integers or slices, not list

In [81]:
data.count()

0     175616
1     175616
2     175616
3     175616
4     175616
       ...  
72    175616
73    175616
74    175616
75    175616
76    175616
Length: 77, dtype: int64

In [83]:
import json
 
with open("userdata.json", "r") as file:
    data = json.load(file)
 

In [161]:

pd.DataFrame(list(data['acquisition_dates'][1:-1].split(",")))

TypeError: tuple indices must be integers or slices, not str

In [133]:
# This is script may only work with sentinelhub.__version__ >= '3.4.0'
def geotiff_multi_caller(bbox_input):
    ''' Returns geotiff and json with NDVi over years
    Args:
    bbox_input - list of bbox [12.44693, 41.870072, 12.541001, 41.917096]
    '''
    import time
    start_time = time.time()
    
    from sentinelhub import SentinelHubRequest, DataCollection, MimeType, CRS, BBox, SHConfig, Geometry
    
    
    # Credentials
    config = SHConfig()
    config.sh_client_id = '96acbb95-6c77-4d93-aaf3-ca218727e1b4'
    config.sh_client_secret = 'F@,ZbT*8PTZ(F}:7@.|,*;#K/K/Ev+#)uRLQ,^t2'
    evalscript = """
    //VERSION=3
    // Script to extract a time series of NDVI values using 
    // Sentinel 2 Level 2A data and  metadata file.
    function setup() {
        return {
          input: [{
            bands: ["B04", "B08"],
            units: "DN"
          }],
          output: {
            bands: 1,
            sampleType: SampleType.FLOAT32
          },
          mosaicking: Mosaicking.ORBIT
        }

      }

      // The following function is designed to update the number of
      // output bands without knowing beforehand how many there are
      function updateOutput(outputs, collection) {
          Object.values(outputs).forEach((output) => {
              output.bands = collection.scenes.length;
          });
      }
      // function to generate a json file with a list of the NDVI 
      // dates used in the analysis. 
      function updateOutputMetadata(scenes, inputMetadata, outputMetadata) {
          var dds = [];
          for (i=0; i<scenes.length; i++){
            dds.push(scenes[i].date)
          }
          outputMetadata.userData = { "acquisition_dates":  JSON.stringify(dds) }
      }

      function evaluatePixel(samples) {
        // Precompute an array to contain NDVI observations
        var n_observations = samples.length;
        let ndvi = new Array(n_observations).fill(0);

        // Fill the array with NDVI values
        samples.forEach((sample, index) => {
          ndvi[index] = (sample.B08 - sample.B04) / (sample.B08 + sample.B04) ;
        });

        return ndvi;
      }
    """
    bbox = BBox(bbox=bbox_input, crs=CRS.WGS84)
    from shapely.geometry import box
    b = box(bbox_input[0],bbox_input[1],bbox_input[2],bbox_input[3])
    points = list(b.exterior.coords)
    polygon = str(points).replace("(","[").replace(")","]")
    #geometry_input = str("""geometry={"type":"Polygon","coordinates":[""")+str(polygon)+str("]}")
    geometry = Geometry(b, crs=CRS.WGS84)
                        
    #geometry = Geometry(geometry={"type":"Polygon","coordinates":[[[-94.04798984527588,41.7930725281021],[-94.04803276062012,41.805773608962866],[-94.06738758087158,41.805901566741305],[-94.06734466552734,41.7967199475024],[-94.06223773956299,41.79144072064381],[-94.0504789352417,41.791376727347966],[-94.05039310455322,41.7930725281021],[-94.04798984527588,41.7930725281021]]]}, crs=CRS.WGS84)

    request = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A,          
                time_interval=('2023-01-01', '2023-12-31'),          
                other_args={"dataFilter": {"maxCloudCoverage": 15,"mosaickingOrder": "leastCC"},"processing": {"harmonizeValues": True}}
            ),
        ],
        responses=[
            SentinelHubRequest.output_response('default', MimeType.TIFF),
        SentinelHubRequest.output_response('userdata', MimeType.JSON),
        ],
        bbox=bbox,
        geometry=geometry,
        size=[779.8034286939699, 523.469],
        config=config
    )

    response = request.get_data()
    
    data = response
    import pickle
    with open('Rome.pickle', 'wb') as f:
        pickle.dump(data, f)
    
    return response, print("--- %s seconds ---" % (time.time() - start_time))

In [134]:
%time
data = geotiff_multi_caller([12.44693, 41.870072, 12.541001, 41.917096])

CPU times: user 4 µs, sys: 1 µs, total: 5 µs
Wall time: 9.06 µs
--- 16.960057973861694 seconds ---


In [120]:
 
#geometry_input = str("""geometry={"type":"Polygon","coordinates":[[""")+str(bbox)+str("]]}")

#create outside geometry polygon

from shapely.geometry import box
bbox_input = [12.44693, 41.870072, 12.541001, 41.917096]
b = box(bbox_input[0],bbox_input[1],bbox_input[2],bbox_input[3])
points = list(b.exterior.coords)
polygon = str(points).replace("(","[").replace(")","]")

In [17]:
data[0][0]["default.tif"]

array([[0.85809904, 0.8396013 , 0.5826087 , ..., 0.40027893, 0.2658228 ,
        0.159799  ],
       [0.8117182 , 0.7386231 , 0.72352284, ..., 0.56177926, 0.33973128,
        0.15959597],
       [0.48040637, 0.44927537, 0.41910022, ..., 0.5772727 , 0.47061422,
        0.2759706 ],
       ...,
       [0.15432526, 0.25096524, 0.15243416, ..., 0.3231323 , 0.31842262,
        0.1255144 ],
       [0.19494204, 0.03623188, 0.19139785, ..., 0.2540132 , 0.47835052,
        0.14594595],
       [0.03035714, 0.18226601, 0.25407925, ..., 0.30110496, 0.6596702 ,
        0.34972447]], dtype=float32)

In [139]:
import pickle
with open('data.pickle', 'wb') as f:
    pickle.dump(data, f)

In [168]:
import bz2
ofile = bz2.BZ2File("data_.pickle",'wb')
pickle.dump(data,ofile)
ofile.close()

In [251]:
import os
print(os.path.getsize("/Users/jbms/TI/Ciechanow/Ciechanow.pickle"))

4889314


In [146]:
import pickle
objects = []
with (open("/Users/jbms/TI/Ciechanow/Ciechanow.pickle", "rb")) as openfile:
    while True:
        try:
            dataciech = pickle.load(openfile)
        except EOFError:
            break


In [147]:
dataciech[0][0]

{'default.tif': array([[[0.0697975 , 0.08059452, 0.07370853, 0.07869561],
         [0.06698669, 0.07280832, 0.07175314, 0.08027523],
         [0.06627361, 0.0755603 , 0.07102665, 0.08223201],
         ...,
         [0.0700021 , 0.07513361, 0.06843456, 0.07760345],
         [0.0666389 , 0.07710011, 0.06308981, 0.07181856],
         [0.06566832, 0.07425641, 0.05776663, 0.06900464]],
 
        [[0.07081902, 0.07814024, 0.06923751, 0.07914748],
         [0.07351998, 0.07761506, 0.06839698, 0.08005763],
         [0.06804337, 0.07742132, 0.07105863, 0.07872663],
         ...,
         [0.0738423 , 0.07823376, 0.06929853, 0.07602825],
         [0.06655198, 0.07979009, 0.06659517, 0.07707068],
         [0.06342924, 0.07377225, 0.06659575, 0.07103242]],
 
        [[0.07592926, 0.08239934, 0.07158029, 0.08023483],
         [0.06976268, 0.07706274, 0.06836018, 0.07953064],
         [0.06635022, 0.07938424, 0.07183345, 0.07469977],
         ...,
         [0.06976237, 0.07884802, 0.06557556, 0.0796

In [156]:
ciechtif = dataciech[0][0]["default.tif"]

In [157]:
import numpy as np
import tifffile

tifffile.imsave('test.tif', ciechtif)

y = tifffile.imread('test.tif')

np.all(np.equal(ciechtif, y)) 

True

In [259]:
import matplotlib.pyplot as plt
I = plt.imread(tif)

AttributeError: 'numpy.ndarray' object has no attribute 'read'

In [260]:
tif

array([[0.85809904, 0.8396013 , 0.5826087 , ..., 0.40027893, 0.2658228 ,
        0.159799  ],
       [0.8117182 , 0.7386231 , 0.72352284, ..., 0.56177926, 0.33973128,
        0.15959597],
       [0.48040637, 0.44927537, 0.41910022, ..., 0.5772727 , 0.47061422,
        0.2759706 ],
       ...,
       [0.15432526, 0.25096524, 0.15243416, ..., 0.3231323 , 0.31842262,
        0.1255144 ],
       [0.19494204, 0.03623188, 0.19139785, ..., 0.2540132 , 0.47835052,
        0.14594595],
       [0.03035714, 0.18226601, 0.25407925, ..., 0.30110496, 0.6596702 ,
        0.34972447]], dtype=float32)

In [150]:
Image.fromarray(tif)

NameError: name 'Image' is not defined

In [226]:
tif

array([[0.85809904, 0.8396013 , 0.5826087 , ..., 0.40027893, 0.2658228 ,
        0.159799  ],
       [0.8117182 , 0.7386231 , 0.72352284, ..., 0.56177926, 0.33973128,
        0.15959597],
       [0.48040637, 0.44927537, 0.41910022, ..., 0.5772727 , 0.47061422,
        0.2759706 ],
       ...,
       [0.15432526, 0.25096524, 0.15243416, ..., 0.3231323 , 0.31842262,
        0.1255144 ],
       [0.19494204, 0.03623188, 0.19139785, ..., 0.2540132 , 0.47835052,
        0.14594595],
       [0.03035714, 0.18226601, 0.25407925, ..., 0.30110496, 0.6596702 ,
        0.34972447]], dtype=float32)

In [262]:
    with open("/Users/jbms/TI/Ciechanow/Ciechanow.tif", "wb") as f:
        f.write(tif)